In [34]:
from tenerife_rag.config import LLMConfig

config = LLMConfig.from_env()
client = config.client()

config.public_dict()

response = client.responses.create(model=config.generation_model, input="Hello, world!", temperature=config.temperature, top_p=config.top_p)
print(response.output_text)

Hello! How can I assist you today?


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from tenerife_rag.chunking import Chunk

pages = PyMuPDFLoader("../data/TENERIFE.pdf").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
windows = splitter.split_documents(pages)

chunks = [
    Chunk(id=str(i), text=w.page_content, pages=[w.metadata["page"] + 1])
    for i, w in enumerate(windows)
]

print(len(chunks))
print(chunks[0].text)
print("-----------------------------------------------")
print(chunks[32].text, chunks[32].citation)

ImportError: cannot import name 'PyMuPDFLoader' from 'langchain_core.document_loaders' (d:\repos\tenerife_rag\.venv\Lib\site-packages\langchain_core\document_loaders\__init__.py)

In [36]:
from tenerife_rag.store import build_store, search

store = build_store(chunks, config.embeddings())

In [37]:
search(store, "playa de arena negra")

[(Chunk(id='8', text='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder igualmente). \n• La Orotava \nEste es el pueblo más bonito y maravilloso del mundo (no es porque yo sea de allí).', kind='ventana', path=[], pages=[8]),
  0.5812820818459936),
 (Chunk(id='25', text='o Playa del Camisón \nEsta es una playa pequeñita que a nosotros en concreto nos gusta mucho.', kind='ventana', path=[], pages=[20]),
  0.5109466661798824),
 (Chunk(id='29', text='• El Médano / La Tejita \nPlayas tipiquísimas de Tenerife y otra de las estampas de la isla con Montaña Roja al \nfondo. Nota: La Tejita es playa nudista para que lo tengáis en cuenta.  \n \n \n• Los Cristianos \nDe la zona de Los Cristianos que me v

In [38]:
search(store, "hay carta en un guachinche")

[(Chunk(id='31', text='atravesaréis una especie de camino de tierra, meteos sin miedo. Si \nvais a Tenerife y no probáis una experiencia en Guachinche, no habéis \nido a Tenerife. Os podría recomendar toda la carta, prácticamente – \nde hecho, no. No podría, ya que en un guachinche no hay carta. Y de \nbebidas solo hay tres cosas: vino, seven up para mezclar con el vino, y \nagua. That’s all. Mi recomendación realmente es que preguntéis a los \ncamareros y, si es vuestra primera experiencia en un guachinche, \nhacédselo saber y que os recomienden ellos para que os vayáis de 10) \n[Tripadvisor] (os podría recomendar más guachinches, pero diría que \neste es mi favorito) \n \nNota: los Guachinches son típicos en la zona norte de Tenerife. \nActualmente existen muchos restaurantes que se llaman Guachinches', kind='ventana', path=[], pages=[24]),
  0.46639605198608325),
 (Chunk(id='32', text='Nota: los Guachinches son típicos en la zona norte de Tenerife. \nActualmente existen muchos resta

In [39]:
INSTRUCTIONS: str = """Eres un guía turístico de Tenerife. Responde en español usando SOLO los fragmentos.
Cita cada dato con el id del fragmento entre corchetes, por ejemplo [S1].
Si los fragmentos no contienen la respuesta, dilo."""

def answer(question:str, top_k: int=3):
    hits = search(store, question, top_k)
    context = "\n".join(
        f"[S{i}] {chunk.citation} {chunk.text}" for i, (chunk, _) in enumerate(hits, 1)
    )
    
    response = client.responses.create(
        model=config.generation_model,
        instructions=INSTRUCTIONS,
        input=f"Fragmentos:\n{context}\n\nPregunta: {question}\n\nRespuesta:",
        temperature=config.temperature,
        top_p=config.top_p,
        max_output_tokens=config.max_output_tokens
    )
    
    return response.output_text, hits

In [40]:
answer("¿Qué playas de arena negra hay en Tenerife?", top_k=3)

('En Tenerife hay playas de arena negra como la Playa de Benijo, que además ofrece atardeceres espectaculares si el día está despejado [S2].',
 [(Chunk(id='29', text='• El Médano / La Tejita \nPlayas tipiquísimas de Tenerife y otra de las estampas de la isla con Montaña Roja al \nfondo. Nota: La Tejita es playa nudista para que lo tengáis en cuenta.  \n \n \n• Los Cristianos \nDe la zona de Los Cristianos que me venga a la mente os recomendaría la Playa de \nLos Cristianos o la Playa de Las Vistas y creo que poco más (la imagen es la Playa de \nLas Vistas).', kind='ventana', path=[], pages=[23]),
   0.6458496728143637),
  (Chunk(id='8', text='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder 